# Converting CAD with specific mesh parameters to neutronics geometry

In this example we have a CAD model with different size components and want to mesh the components so that the:
- meshed geometry is accurately represents the CAD geometry, i.e. the volume of the curved CAD is very close to the mesh volume.
- the mesh uses minimal number of triangles (as lots of triangles can slow the simulation down)

We start by making a geometry that has flat surfaces (these require very few mesh elements) and curved surfaces (which require more mesh elements).

In [ ]:
import cadquery as cq
import cad_to_dagmc
import gmsh
import assembly_mesh_plugin


In [ ]:

box_shape1 = cq.Workplane("XY").box(50, 50, 50)
box_shape2 = cq.Workplane("XY").moveTo(0, 50).box(50, 50, 100)


In [ ]:

assembly = cq.Assembly()
assembly.add(box_shape1, name="first_material")
assembly.add(box_shape2, name="second_material")

In [ ]:
# getTaggedGmsh initializes gmsh and creates a mesh object ready for meshing
assembly.getTaggedGmsh()
# Here you can set the mesh parameters
# In this case, we set the minimum and maximum mesh size
# but you can set any other Gmsh parameters
# Remember that the gmsh has physical groups if you want to use them when meshing
gmsh.option.setNumber("Mesh.MeshSizeMax", 4.2)
gmsh.model.mesh.generate(2)  # for DAGMC surface mesh we just need a 2D surface mesh


In [ ]:

cad_to_dagmc.export_gmsh_object_to_dagmc_h5m_file(filename="dagmc_from_gmsh_object.h5m")

# finalize the GMSH API after using export_gmsh_object_to_dagmc_h5m_file
# and getTaggedGmsh as these both need access to the GMSH object.
gmsh.finalize()

Now we have a DAGMC geometry with the meshing done to the user specifications.

This h5m file can be used in DAGMC enabled OpenMC simulations